# Reproducibility in Deep Learning
> do you want to check your ideas in DL? you need Reproducibility (PyTorch, TF2.X)
- toc: true
- branch: master
- badges: true,
- comments: true
- image: images/reproducibility.jpg
- author: Sajjad Ayoubi
- categories: [tips]

# Reproducibility ?!
- deep learning training processes are stochastic in nature,
During development of a model, sometimes it is useful to be able to obtain reproducible results from run to run in order to determine if a change in performance is due to an actual model or data modification, also for comparing different things and evaluate new tricks and ideas
we need to train our neural nets in a deterministic way
- In the process of training a neural network, there are multiple stages where randomness is used, for example

  - random initialization of weights of the network before the training starts.
  - regularization, dropout, which involves randomly dropping nodes in the network while training.
  - optimization process like SGD or Adam also include random initializations.

- we will see that how can we use Frameworks in a deterministic way
- note in deterministic training you are a bit slow than stochastic

# PyTorch
- Mnist classification with Reproducibility
> from PyTorch Team: Completely reproducible results are not guaranteed across PyTorch releases, individual commits, or different platforms. Furthermore, results may not be reproducible between CPU and GPU executions, even when using identical seeds, also Deterministic operations are often slower than nondeterministic operations



- the following works with all models (maybe not LSTMs I didn’t check that)

In [ ]:
import numpy as np
import random, os

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms

- create dataloder

In [ ]:
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0), (255))])
train_ds = datasets.MNIST(root='./data', train=True, download=True, transform=transform)

# if you set augmentations set worker_init_fn=(random.seed(0)) and num_workers=0 in dataloder
train_dl = torch.utils.data.DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=4)

- the following works with all models

In [ ]:
def torch_seed(seed=0):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.cuda.manual_seed_all(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)

In [ ]:
def train(reproducibility=True, n_run=2, device='cuda'):
    
    for n in range(n_run):
      print('run number: ', n+1)

      # set seed before create your model  
      if reproducibility:
          torch_seed(seed=0)
      # compile model
      model = nn.Sequential(nn.Flatten(), nn.Linear(28*28, 128), nn.GELU(), nn.Linear(128, 10)).to(device)
      loss_fn = nn.CrossEntropyLoss().to(device)
      optimizer = optim.AdamW(model.parameters(), lr=0.005, weight_decay=0.0)
      # training loop
      loss_avg = 0.0
      for i, data in enumerate(train_dl):
          inputs, labels = data
          optimizer.zero_grad()
          outputs = model(inputs.to(device))
          loss = loss_fn(outputs, labels.to(device))
          loss_avg = (loss_avg * i + loss) / (i+1)
          loss.backward()
          optimizer.step()
          if i%850==0:   
              print('[%d, %4d] loss: %.4f' %(i+1, len(train_dl), loss_avg))

In [ ]:
train(reproducibility=False)

run number:  1
[1, 1875] loss: 2.2943
[851, 1875] loss: 0.8099
[1701, 1875] loss: 0.5946
run number:  2
[1, 1875] loss: 2.2945
[851, 1875] loss: 0.8078
[1701, 1875] loss: 0.5921


In [ ]:
train(reproducibility=True)

run number:  1
[1, 1875] loss: 2.2983
[851, 1875] loss: 0.8051
[1701, 1875] loss: 0.5927
run number:  2
[1, 1875] loss: 2.2983
[851, 1875] loss: 0.8051
[1701, 1875] loss: 0.5927


- if you check your new ideas like me
- you have to always see how much is overhead of your implementation
- in pytorch for giving acutual time we use `synchronize`

In [ ]:
%%timeit
# stay in GPUs until it done
torch.cuda.synchronize()

# Keras & TF 2.X
- Mnist classification with Reproducibility
> from Keras Team: when running on a GPU, some operations have non-deterministic outputs, in particular tf.reduce_sum(). This is due to the fact that GPUs run many operations in parallel, so the order of execution is not always guaranteed. Due to the limited precision of floats, even adding several numbers together may give slightly different results depending on the order in which you add them. You can try to avoid the non-deterministic operations, but some may be created automatically by TensorFlow to compute the gradients, so it is much simpler to just run the code on the CPU. For this, you can set the CUDA_VISIBLE_DEVICES environment variable to an empty string

- they said Keras REPRODUCIBILITY works just on CPUs
- but we need GPUs
- after a week seach I found a possible way on GPUs
  - based on this work [TensorFlow Determinism](https://github.com/NVIDIA/framework-determinism) from `NVIDIA`
  - now we can run Keras with REPRODUCIBILITY on GPUs :)

- Note: it works just for `TF >= 2.3`
  - also it works fine with `tf.data`
  - but you have to watch out (especially prefetch) 

- let's check this out

In [24]:
import random, os
import numpy as np
from tensorflow.keras import layers, models
import tensorflow as tf
from tensorflow.keras.layers import Dense, Flatten,Conv2D,MaxPool2D
from tensorflow.keras.datasets import cifar10,mnist
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.utils import to_categorical
import matplotlib.pyplot as plt
import numpy as np
import random 
import os

In [57]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
x_train, x_test = x_train / 255.0, x_test / 255.0

In [46]:
def tf_seed(seed=0):
	os.environ['PYTHONHASHSEED'] = str(seed)
	# For working on GPUs from "TensorFlow Determinism"
	os.environ["TF_DETERMINISTIC_OPS"] = str(seed)
	np.random.seed(seed)
	random.seed(seed)
	tf.random.set_seed(seed)

In [114]:
def augment(x_train,x_test):
        width_shift = 3/32
        height_shift = 3/32
        flip = True
        datagen = ImageDataGenerator(
            horizontal_flip=flip,
            width_shift_range=width_shift,
            height_shift_range=height_shift,
            )
        datagen.fit(x_train)
        return datagen

In [54]:
def reshape(x_train,x_test):
  # reshape to be [samples][width][height][channels]
        x_train = x_train.reshape((x_train.shape[0], 28, 28, 1))
        x_test = x_test.reshape((x_test.shape[0], 28, 28, 1))
        # convert from int to float
        x_train = x_train.astype('float32')
        x_test = x_test.astype('float32')
        return x_train,x_test

In [115]:
def train(reproducibility=True, n_run=2,aug=False):
    
    for n in range(n_run):
      print('run number: ', n+1)

      # set seed before create your model  
      if reproducibility:
          tf_seed(seed=0)

     # data_augmentation = tf.keras.Sequential([layers.RandomFlip("horizontal_and_vertical"), layers.RandomRotation(0.2),])
      data_augmentation = tf.keras.Sequential([ layers.RandomRotation(0.9)])
      # compile model
      model = tf.keras.models.Sequential([data_augmentation,Conv2D(28, (3, 3), activation='gelu', padding='same', input_shape=(28, 28, 1)),Conv2D(28, (3, 3), activation='gelu', padding='same'),MaxPool2D((2,2)),Flatten(), Dense(128, activation='gelu'), Dense(10, activation='softmax')])
      
      loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
      model.compile(optimizer='adam', loss=loss_fn,metrics=['accuracy'])
      # training 
      if aug:
        print("iam heers")
        datagen=augment(x_train,x_test)
        model.fit(datagen.flow(x_train, y_train, batch_size=batch_size),epochs=1, validation_data=(x_test, y_test))
      else:
        model.fit(x_train, y_train, epochs=1,validation_data=(x_test, y_test))

In [111]:
 x_train,x_test=reshape(x_train,x_test)
 train(reproducibility=False)

run number:  1


/usr/local/lib/python3.7/dist-packages/tensorflow/python/util/dispatch.py:1096: UserWarning: "`sparse_categorical_crossentropy` received `from_logits=True`, but the `output` argument was produced by a sigmoid or softmax activation and thus does not represent logits. Was this intended?"
  return dispatch_target(*args, **kwargs)


1875/1875 [==============================] - 25s 13ms/step - loss: 0.6511 - accuracy: 0.7854 - val_loss: 0.3772 - val_accuracy: 0.8825
run number:  2
1875/1875 [==============================] - 25s 13ms/step - loss: 0.6351 - accuracy: 0.7907 - val_loss: 0.3891 - val_accuracy: 0.8737
run number:  3
1875/1875 [==============================] - 25s 12ms/step - loss: 0.6538 - accuracy: 0.7853 - val_loss: 0.4166 - val_accuracy: 0.8714
run number:  4
1875/1875 [==============================] - 25s 13ms/step - loss: 0.6436 - accuracy: 0.7873 - val_loss: 0.4127 - val_accuracy: 0.8714
run number:  5
1875/1875 [==============================] - 25s 13ms/step - loss: 0.6507 - accuracy: 0.7849 - val_loss: 0.3384 - val_accuracy: 0.8927
run number:  6
1875/1875 [==============================] - 25s 13ms/step - loss: 0.6504 - accuracy: 0.7854 - val_loss: 0.3611 - val_accuracy: 0.8836


In [ ]:
 x_train,x_test=reshape(x_train,x_test)
train(reproducibility=True,aug=True)


run number:  1
iam heers


/usr/local/lib/python3.7/dist-packages/tensorflow/python/util/dispatch.py:1096: UserWarning: "`sparse_categorical_crossentropy` received `from_logits=True`, but the `output` argument was produced by a sigmoid or softmax activation and thus does not represent logits. Was this intended?"
  return dispatch_target(*args, **kwargs)


1875/1875 [==============================] - 35s 18ms/step - loss: 1.0353 - accuracy: 0.6403 - val_loss: 1.1064 - val_accuracy: 0.6361
run number:  2
iam heers
 512/1875 [=======>......................] - ETA: 23s - loss: 1.5254 - accuracy: 0.4564

- if you want run it on CPUs see this

In [ ]:
def tf_seed(seed=0):
    os.environ['PYTHONHASHSEED'] = str(seed)
    # if your machine has GPUs use following to off it
    os.environ['CUDA_VISBLE_DEVICE'] = ''
    np.random.seed(seed)
    random.seed(seed)
    python_random.seed(seed)
    tf.random.set_seed(seed)

In [17]:
# reshape to be [samples][width][height][channels]
x_train = x_train.reshape((x_train.shape[0], 28, 28, 1))
x_test = x_test.reshape((x_test.shape[0], 28, 28, 1))
# convert from int to float
x_train = x_train.astype('float32')
x_test = x_test.astype('float32')

In [18]:
batch_size=32
epochs=2

 # training 
m_no_aug= train(reproducibility=True)
history_no_aug = m_no_aug.fit(
      x_train, y_train,
      epochs=epochs, batch_size=batch_size,
      validation_data=(x_test, y_test))

run number:  1
Epoch 1/2


/usr/local/lib/python3.7/dist-packages/tensorflow/python/util/dispatch.py:1096: UserWarning: "`sparse_categorical_crossentropy` received `from_logits=True`, but the `output` argument was produced by a sigmoid or softmax activation and thus does not represent logits. Was this intended?"
  return dispatch_target(*args, **kwargs)


1875/1875 [==============================] - 22s 11ms/step - loss: 0.1197 - accuracy: 0.9653 - val_loss: 0.0436 - val_accuracy: 0.9851
Epoch 2/2
1875/1875 [==============================] - 21s 11ms/step - loss: 0.0370 - accuracy: 0.9883 - val_loss: 0.0410 - val_accuracy: 0.9869


In [14]:


# width_shift = 3/32
# height_shift = 3/32

it = datagen.flow(x_train, y_train, shuffle=False)
batch_images, batch_labels = next(it)
# visualize_data(batch_images, batch_labels, class_names)

In [16]:
batch_size=32
epochs=2

 # training 
m_aug= train(reproducibility=True)
history_aug = m_aug.fit(
    datagen.flow(x_train, y_train, batch_size=batch_size),
    epochs=epochs,
    validation_data=(x_test, y_test))

run number:  1
Epoch 1/2


/usr/local/lib/python3.7/dist-packages/tensorflow/python/util/dispatch.py:1096: UserWarning: "`sparse_categorical_crossentropy` received `from_logits=True`, but the `output` argument was produced by a sigmoid or softmax activation and thus does not represent logits. Was this intended?"
  return dispatch_target(*args, **kwargs)


1875/1875 [==============================] - 21s 11ms/step - loss: 0.2019 - accuracy: 0.9363 - val_loss: 0.0848 - val_accuracy: 0.9703
Epoch 2/2
1875/1875 [==============================] - 19s 10ms/step - loss: 0.0705 - accuracy: 0.9781 - val_loss: 0.0709 - val_accuracy: 0.9765


In [100]:
def plot_images(dataset, n_images, samples_per_image):
    output = np.zeros((32 * n_images, 32 * samples_per_image, 3))

    row = 0
    for images in dataset.repeat(samples_per_image).batch(n_images):
        output[:, row*32:(row+1)*32] = np.vstack(images.numpy())
        row += 1

    plt.figure()
    plt.imshow(output)
    plt.show()

def flip(x: tf.Tensor) -> tf.Tensor:
    """Flip augmentation

    Args:
        x: Image to flip

    Returns:
        Augmented image
    """
    x = tf.image.random_flip_left_right(x)
    x = tf.image.random_flip_up_down(x)

    return x

def color(x: tf.Tensor) -> tf.Tensor:
    """Color augmentation

    Args:
        x: Image

    Returns:
        Augmented image
    """
    x = tf.image.random_hue(x, 0.08)
    x = tf.image.random_saturation(x, 0.6, 1.6)
    x = tf.image.random_brightness(x, 0.05)
    x = tf.image.random_contrast(x, 0.7, 1.3)
    return x

def rotate(x: tf.Tensor) -> tf.Tensor:
    """Rotation augmentation

    Args:
        x: Image

    Returns:
        Augmented image
    """

    return tf.image.rot90(x, tf.random.uniform(shape=[], minval=0, maxval=4, dtype=tf.int32))

def zoom(x: tf.Tensor) -> tf.Tensor:
    """Zoom augmentation

    Args:
        x: Image

    Returns:
        Augmented image
    """

    # Generate 20 crop settings, ranging from a 1% to 20% crop.
    scales = list(np.arange(0.8, 1.0, 0.01))
    boxes = np.zeros((len(scales), 4))

    for i, scale in enumerate(scales):
        x1 = y1 = 0.5 - (0.5 * scale)
        x2 = y2 = 0.5 + (0.5 * scale)
        boxes[i] = [x1, y1, x2, y2]

    def random_crop(img):
        # Create different crops for an image
        crops = tf.image.crop_and_resize([img], boxes=boxes, box_ind=np.zeros(len(scales)), crop_size=(32, 32))
        # Return a random crop
        return crops[tf.random.uniform(shape=[], minval=0, maxval=len(scales), dtype=tf.int32)]


    choice = tf.random.uniform(shape=[], minval=0., maxval=1., dtype=tf.float32)

    # # Only apply cropping 50% of the time
    # return tf.cond(choice < 0.5, lambda: x, lambda: random_crop(x))


In [101]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

x_train = (x_train/ 255).astype(np.float32)
x_test = (x_test/ 255).astype(np.float32)
x_train,x_test=reshape(x_train,x_test)

In [102]:

dataset = tf.data.Dataset.from_tensor_slices(x_train)

# Add augmentations
augmentations = [flip, color, zoom, rotate]

for f in augmentations:
    dataset = dataset.map(lambda x: tf.cond(tf.random.uniform([], 0, 1) > 0.75, lambda: f(x), lambda: x), num_parallel_calls=4)
dataset = dataset.map(lambda x: tf.clip_by_value(x, 0, 1))

plot_images(dataset, n_images=8, samples_per_image=10)

TypeError: ignored

In [84]:
tf.random.uniform(shape=[2])

<tf.Tensor: shape=(2,), dtype=float32, numpy=array([0.01353359, 0.09272444], dtype=float32)>